# Stock Data Extraction via Web Scraping

Not all financial data is accessible through APIs. This notebook demonstrates how to extract historical stock price tables directly from web pages using `BeautifulSoup` and `requests`. We scrape structured HTML tables and transform them into clean, analysis-ready Pandas DataFrames.

**Stocks Covered:** Netflix (NFLX) · Amazon (AMZN)  
**Tools:** Python · BeautifulSoup · Requests · Pandas

## Table of Contents
1. [Setup & Imports](#setup)
2. [Netflix (NFLX) — Web Scraping Walkthrough](#netflix)
   - Fetching & Parsing HTML
   - Extracting the Price Table
   - Quick Extraction with `pd.read_html`
3. [Amazon (AMZN) — Applied Scraping](#amazon)
4. [Key Findings](#findings)


In [1]:
# Install required libraries (run once)
!pip install pandas requests bs4 html5lib lxml --quiet

## 1. Setup & Imports <a id='setup'></a>

In [2]:
import warnings
import pandas as pd
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Netflix (NFLX) — Web Scraping Walkthrough <a id='netflix'></a>

We use Netflix as our primary example to walk through the complete web scraping pipeline, from sending an HTTP request to producing a clean DataFrame.

### Step 1 — Fetch the Web Page

We send an HTTP GET request to retrieve the raw HTML content of the target page.

In [3]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/netflix_data_webpage.html"
data = requests.get(url).text
print(f"Page fetched successfully — {len(data):,} characters")

Page fetched successfully — 684,271 characters


### Step 2 — Parse HTML with BeautifulSoup

BeautifulSoup converts the raw HTML string into a navigable tree structure, allowing us to locate and extract specific elements.

In [4]:
soup = BeautifulSoup(data, 'html.parser')
print("Page title:", soup.title.string)

Page title: Netflix, Inc. (NFLX) Stock Historical Prices & Data - Yahoo Finance


### Step 3 — Extract the Price Table

We navigate to the `<tbody>` tag, iterate through each row `<tr>`, and collect cell values `<td>`. The result is loaded into a Pandas DataFrame with columns: Date, Open, High, Low, Close, Adj Close, Volume.

In [5]:
netflix_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"])

for row in soup.find("tbody").find_all('tr'):
    col = row.find_all("td")
    if len(col) < 7:
        continue
    netflix_data = pd.concat([netflix_data, pd.DataFrame({
        "Date":      [col[0].text],
        "Open":      [col[1].text],
        "High":      [col[2].text],
        "Low":       [col[3].text],
        "Close":     [col[4].text],
        "Adj Close": [col[5].text],
        "Volume":    [col[6].text]
    })], ignore_index=True)

print(f"Extracted {len(netflix_data)} rows of Netflix price data")
netflix_data.head()

Extracted 70 rows of Netflix price data


,Date,Open,High,Low,Close,Adj Close,Volume
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,528.21,"78,560,600"
1,"May 01, 2021",512.65,518.95,478.54,502.81,502.81,"66,927,600"
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,513.47,"111,573,300"
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,521.66,"90,183,900"
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,538.85,"61,902,300"


### Alternative: Quick Extraction with `pd.read_html`

`pd.read_html()` can detect and parse HTML tables automatically — ideal for well-structured pages. It's faster than manual scraping but offers less control over edge cases.

In [6]:
netflix_quick = pd.read_html(url)[0]
print(f"pd.read_html extracted {len(netflix_quick)} rows")
netflix_quick.head()

pd.read_html extracted 71 rows


,Date,Open,High,Low,Close*,Adj Close**,Volume
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,528.21,78560600
1,"May 01, 2021",512.65,518.95,478.54,502.81,502.81,66927600
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,513.47,111573300
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,521.66,90183900
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,538.85,61902300


## 3. Amazon (AMZN) — Applied Scraping <a id='amazon'></a>

We now apply the same scraping pipeline to Amazon's historical stock data, demonstrating how the approach generalises across different pages.

In [7]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/amazon_data_webpage.html"
html_data = requests.get(url).text
soup = BeautifulSoup(html_data, "html.parser")
print("Page title:", soup.title.string)

Page title: Amazon.com, Inc. (AMZN) Stock Historical Prices & Data - Yahoo Finance


In [8]:
amazon_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"])

for row in soup.find("tbody").find_all("tr"):
    col = row.find_all("td")
    if len(col) < 7:
        continue
    amazon_data = pd.concat([amazon_data, pd.DataFrame({
        "Date":      [col[0].text],
        "Open":      [col[1].text],
        "High":      [col[2].text],
        "Low":       [col[3].text],
        "Close":     [col[4].text],
        "Adj Close": [col[5].text],
        "Volume":    [col[6].text]
    })], ignore_index=True)

print(f"Extracted {len(amazon_data)} rows of Amazon price data")
print(f"Columns: {list(amazon_data.columns)}")
print(f"Opening price on last recorded date: {amazon_data.tail(1)['Open'].values[0]}")
amazon_data.head()

Extracted 61 rows of Amazon price data
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
Opening price on last recorded date: 656.29


,Date,Open,High,Low,Close,Adj Close,Volume
0,"Jan 01, 2021","3,270.00","3,363.89","3,086.00","3,206.20","3,206.20","71,528,900"
1,"Dec 01, 2020","3,188.50","3,350.65","3,072.82","3,256.93","3,256.93","77,556,200"
2,"Nov 01, 2020","3,061.74","3,366.80","2,950.12","3,168.04","3,168.04","90,810,500"
3,"Oct 01, 2020","3,208.00","3,496.24","3,019.00","3,036.15","3,036.15","116,226,100"
4,"Sep 01, 2020","3,489.58","3,552.25","2,871.00","3,148.73","3,148.73","115,899,300"


## 4. Key Findings <a id='findings'></a>

This notebook demonstrated how to extract financial data from web pages when no API is available:

- **Web scraping as a data source:** HTML tables on finance pages can be reliably parsed using BeautifulSoup, providing access to historical data that isn't available through standard APIs.
- **Manual vs automated extraction:** The step-by-step BeautifulSoup approach gives full control over column selection and row filtering, while `pd.read_html()` provides a faster shortcut for clean, well-structured pages.
- **Reusability:** The same scraping pipeline works across both Netflix and Amazon with no structural changes, confirming that the approach generalises well across similarly formatted financial data pages.
- **Data quality considerations:** Raw scraped data often contains formatting artifacts (commas, currency symbols, percentage signs) that must be cleaned before any quantitative analysis can be performed.
